# Transition design examples

Taper/transition designs between waveguide widths (and, in the dual-layer case, between
two process layers), evaluated with a Lumerical MODE EME simulation.

## Step 0: Open the Lumerical MODE session

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import lumapi
import numpy as np
import matplotlib.pyplot as plt

import lumapi_util as lu
from lumapi_util.paths import STACK_DATA_DIR

sim = lumapi.MODE()
backend = lu.LumericalModeBackend(sim)

## Shared geometry and sweep helpers

`build_transition` builds a taper geometry from a list of per-layer specs (one entry for a
single-layer transition, two entries for a dual-layer transition) and sets up the EME
simulation. `taper_length_sweep` sweeps the taper length of a given EME cell group and reports
the excess loss.

In [ ]:
# Defaults for build_transition; override any of these by passing them as keyword arguments
default_transition_params = {
    "wavelength": 1.55,
    "sim_size": (3, 3),
    "n_modes": 10,
    "mesh_steps": (0.02, 0.02),
    "bc": ["Anti-Symmetric", "PML", "PML", "PML"],
    "eme_offset": 1.0,
    "profile_size": (60, 20),
}


def build_transition(backend, layers, **params):
    """Set up an EME taper/transition from one or more per-layer geometry specs."""
    params = {**default_transition_params, **params}

    backend.switch_to_layout()
    backend.clear_layer_geometries()

    lengths = [layer["length"] for layer in layers]
    x_offsets = [layer.get("x0", 0.0) for layer in layers]
    widths = []
    layer_infos = []

    for layer in layers:
        wi, wf, length = layer["wi"], layer["wf"], layer["length"]
        extension = layer.get("extension", (0.0, 0.0))
        profile = layer.get("profile", "linear")
        offset = np.array([layer.get("x0", 0.0), layer.get("y0", 0.0)])

        backend.configure_layer(layer["layer"], sidewall_angle=layer.get("sidewall_angle", 90))
        taper = lu.build_profile_taper(wi, wf, length, extension=extension, profile=profile) + offset
        backend.set_layer_geometries(layer["layer"], [taper])

        widths.extend([wi, wf])
        layer_infos.append(backend.get_layer_info(layer["layer"]))

    offset = max(x_offsets, default=0.0)
    shortest_length, longest_length = min(lengths), max(lengths)
    cell_lengths = [1.0, offset, shortest_length, longest_length - shortest_length, 1.0]
    eme_cells = [(int(length), length) for length in cell_lengths if length > 0]

    sim_width, sim_height = params["sim_size"]
    z0, thickness = layer_infos[0]["z0"], layer_infos[0]["t"]
    zc = z0 + thickness / 2
    eme_offset = params["eme_offset"]

    backend.configure_eme(
        size=[None, sim_width, sim_height],
        center=[eme_offset, 0.0, zc],
        cells=eme_cells,
        bc=params["bc"],
        wavelength=params["wavelength"],
        n_modes=params["n_modes"],
    )
    backend.configure_mesh(
        name="mesh_struct",
        size=[sum(cell_lengths) + 2, max(widths), thickness],
        center=[sum(cell_lengths) / 2 + eme_offset, 0.0, zc],
        steps=[None, *params["mesh_steps"]],
    )

    profile_x, profile_span = params["profile_size"]
    backend.configure_eme_profile(size=[profile_x, profile_span, None], center=[profile_x / 2, 0.0, zc])
    backend.configure_eme_profile(size=[profile_x, None, profile_span / 2], center=[profile_x / 2, 0.0, zc])


def compute_excess_loss(backend):
    scattering_matrix = backend.eme_propagation()
    power = 2 * lu.lin2db(scattering_matrix)
    return -power[1, 0]


def set_eme_group_lengths(backend, cell_range, length):
    return lu.set_eme_group_lengths(backend.sim, cell_range, length)


def taper_length_sweep(backend, cell_range=1, lengths=None, plot=True, filename="taper_length_sweep.csv"):
    if lengths is None:
        lengths = np.arange(1, 30, 1)

    excess_loss, summary = lu.sweep_param_result(
        backend,
        sweep_params={"length": lengths},
        geometry_params={"cell_range": cell_range},
        geometry_fn=set_eme_group_lengths,
        result_fn=compute_excess_loss,
        result_type=float,
        save_path=filename,
    )

    if plot:
        lu.plot_metric_1d({"length": lengths}, excess_loss, x_param="length", metric_name="Excess Loss")

    return summary

## Example 1: single-layer transition in Si at 1550 nm

Taper a single silicon waveguide layer from a narrow to a wider width, using the generic
sample SOI stack.

In [ ]:
backend.clear_geometry()

stack_file = STACK_DATA_DIR / "sample_soi.lbr"
stack_length = 100
backend.load_layer_stack(stack_file, size=[stack_length, 20, None])

si_transition = [
    {"layer": "WG", "wi": 0.4, "wf": 0.9, "length": 20, "extension": (2, 2)},
]

si_transition_setup = partial(build_transition, layers=si_transition, wavelength=1.55)
si_transition_setup(backend)

In [ ]:
backend.run_eme("si_transition_1550nm.lms")

In [ ]:
# cell_range = backend.expand_eme_cell_group(3)
lengths = np.arange(1, 50, 1)
EL = taper_length_sweep(backend, cell_range=1, lengths=lengths, filename="si_transition_1550nm_sweep.csv")

## Example 2: single-layer transition in SiN at 632 nm

Same taper approach, this time in the silicon-nitride layer at a shorter wavelength.

In [ ]:
sim.switchtolayout()
sim.deleteall()

stack_file = STACK_DATA_DIR / "sample_sin.lbr"
stack_length = 100
lu.add_layerstack_fromfile(sim, filename=stack_file.as_posix(), size=[stack_length, 20, None])

sin_transition = [
    {"layer": "WG", "wi": 0.3, "wf": 0.8, "length": 20, "extension": (2, 2)},
]

build_transition(sim, sin_transition, wavelength=0.632)

In [ ]:
lu.run_eme(sim, "sin_transition_632nm.lms")

In [ ]:
# cell_range = lu.expand_cell_group(sim, 3)
EL = taper_length_sweep(sim, cell_range=2, filename="sin_transition_632nm_sweep.csv")

## Example 3: dual-layer transition, SiN to Si at 1550 nm

Transition optical power from a SiN routing layer down into a Si device layer using a new
dual-layer stack (`sample_soi_sin.lbr`) that combines both a Si and a SiN layer, vertically
separated by a spacer oxide. The SiN taper narrows down while the Si taper (offset below it)
widens up, transferring the mode between layers.

In [ ]:
sim.switchtolayout()
sim.deleteall()

stack_file = STACK_DATA_DIR / "sample_soi_sin.lbr"
stack_length = 30
lu.add_layerstack_fromfile(sim, filename=stack_file.as_posix(), size=[stack_length, 20, None])

dual_transition = [
    {"layer": "WG_SIN", "wi": 1.0, "wf": 0.2, "length": 20, "extension": (2, 0)},
    {"layer": "WG_SI", "wi": 0.2, "wf": 0.4, "length": 20, "x0": 2, "extension": (0, 2)},
]

build_transition(sim, dual_transition, wavelength=1.55)

In [ ]:
lu.run_eme(sim, "dual_transition_sin_to_si_1550nm.lms")

In [ ]:
cell_range = lu.expand_cell_group(sim, 3)
EL = taper_length_sweep(sim, cell_range=cell_range, filename="dual_transition_sin_to_si_1550nm_sweep.csv")